# Chapter 17: Monitoring, Logging and Debugging (notebook edition)

## Learning Objectives

- Apply the annotation cap
- Read a debug trace and a re-run
- Compute metrics from run history

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. The annotation cap

Fifteen emitted per level in one step; the cap is ten per level per step, so two steps can surface more.

In [ ]:
import json
s = json.loads((ROOT/"fixtures"/"observability_ch17_summary.json").read_text())
surfaced = lambda n, cap=10: min(n, cap)
a = s["annotations"]
print(a["surfaced"], "| 2 steps x 8 errors ->", a["two_steps_of_8_errors"]["total"])
assert surfaced(15) == 10 and surfaced(25) + surfaced(25) == 20 and a["two_steps_of_8_errors"]["total"] == 16

## 2. `rerun --failed` re-executes only failed jobs

`stable` printed `stable_attempt=1` at attempt 2: its result was carried over.

In [ ]:
rr = s["rerun"]["attempt2_via_rerun_failed"]
print(rr)
assert rr["flaky_attempt_value"] == 2 and rr["stable_attempt_value"] == 1 and not rr["commands_started_again"]

## 3. Debug logging and the expression trace

The debug re-run added 140 lines; the trace shows each expression with values substituted. Our evaluator reproduces the expanded result.

In [ ]:
d = s["debug"]
print(d["attempt1"], d["attempt3_via_rerun_debug"])
for line in d["trace_example"]: print(line)
from intro_gha.expr import evaluate, stringify
print(stringify(evaluate("github.event_name == 'workflow_dispatch' && true", {"github": {"event_name": "workflow_dispatch"}})))
assert d["attempt1"]["debug_lines"] == 0 and d["attempt3_via_rerun_debug"]["debug_lines"] == 140

## 4. act's stages equal our waves

`act -l` printed dependency stages; `execution_waves` computes the same.

In [ ]:
from intro_gha.workflow import execution_waves
g = {"a": [], "b": ["a"], "c": ["b"], "report_always": ["a","b","c"], "report_on_failure": ["a","b","c"], "report_default": ["a"]}
waves = execution_waves(g); print(waves)
assert {str(i): w for i, w in enumerate(waves)} == s["act"]["list_stages"]

## 5. Metrics from this repository's own history

178 runs; the 11 failed CI runs were all preventable locally.

In [ ]:
import collections
snap = json.loads((ROOT/"fixtures"/"runs_ch17_snapshot.json").read_text()); runs = snap["runs"]
c = collections.Counter(r["conclusion"] for r in runs)
print(dict(c), f"success {100*c['success']/len(runs):.1f}%")
ci = [r for r in runs if r["workflow"] == "ci.yml"]; ok = sum(r["conclusion"] == "success" for r in ci)
fails = json.loads((ROOT/"fixtures"/"ci_failures_ch17.json").read_text())
print("ci.yml:", len(ci), "runs,", ok, "ok |", collections.Counter(f["cause"] for f in fails))
assert (len(ci), ok, len(fails)) == (57, 46, 11)

## Takeaways & Next Steps

- Ten annotations per level per step; use summaries for the rest.
- Debug re-runs show how expressions expanded.
- Classify failures before calling them flaky.
- Next: Chapter 18, cost, performance and limits.

## Chapter Link

Read: `learning_modules/chapter_17_monitoring_debugging.md`